In [12]:
# ==========================================
# 삼성전자 주가 예측 (LSTM)
# ==========================================

!pip install yfinance

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf

from sklearn.preprocessing import MinMaxScaler

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Input, SimpleRNN
from tensorflow.keras.callbacks import EarlyStopping


In [ ]:
df = yf.download(
    "005930.KS",
    start="2018-01-01",
    end="2026-07-01",
    auto_adjust=True
)

df.head()

In [ ]:
close = df[["Close"]]

close.info()

In [ ]:
# 정규화 

# 최소값 = 100
# 최대값 = 190

# 현재값−100
#------------
#   190−100
scaler = MinMaxScaler() # StandardScaler
scaled = scaler.fit_transform(close)

scaled

In [ ]:
# 시계열 데이터 생성
# 60일 -> 다음날

# ==========================================
# data_size = -1
# time_steps = 60
# features = 1
# ==========================================

window_size = 60

x = []
y = []

for i in range(window_size,len(scaled)):
    x.append(scaled[i - window_size : i])
    y.append(scaled[i])
    #print(window_size,i)

x = np.array(x)
y = np.array(y)

#(2020, 60, 1) (2020, 1)
print(x.shape, y.shape)

In [11]:
# train/ test 테이터 나누기

train_size = int(len(x) * 0.8)
train_size

x_train = x[:train_size]
x_test = x[train_size:]

y_train = y[:train_size]
y_test = y[train_size:]

print(x_train.shape)
print(x_test.shape)


(1616, 60, 1)
(404, 60, 1)


In [ ]:
# 모델링

model = Sequential([
    Input(shape=(window_size,1)),
    SimpleRNN(64,return_sequences=True),
    SimpleRNN(32),
    Dense(16,activation="relu"),
    Dense(1)
])

# model = Sequential([
#     Input(shape=(window_size,1)),
#     LSTM(64,return_sequences=True),
#     LSTM(32),
#     Dense(16,activation="relu"),
#     Dense(1)
# ])

model.compile(
    optimizer="adam",
    loss ="mse", 
    metrics=["mae"] # 평균 절대 오차 # 선형회귀 이기 때문에 accurcy 아님
)